# Prediction explanations

**SERS-D2DNet + SuperRaman · v0.3.1**

[Notebook guide](../docs/NOTEBOOK_GUIDE.md) · [Paper alignment](../docs/PAPER_IMPLEMENTATION_MAP.md) · [README](../README.md)

## Goal
Explain a trained run's classifier on explicitly selected original or translated test rows.

## Setup
Public notebooks start without saved execution outputs. Select the Python environment where `.[notebooks]` is installed. Edit `CONFIG` below to select the demo or your research configuration. Every notebook opens the same persistent run. Configuration changes require a new `run_directory`. The default executes a synthetic, one-epoch, width-2 demonstration across all four portable devices.

In [ ]:
CONFIG = "configs/workflow_demo.json"
RUN_RESEARCH_TRAINING = False

In [ ]:
import numpy as np
from sersd2d_superraman import __version__
from sersd2d_superraman.provenance import read_json
from sersd2d_superraman.data import load_batch
from sersd2d_superraman.notebook import (
    open_workflow,
    display_table,
    setup_plotting,
    training_plot,
    spectrum_plot,
    confusion_plot,
)

workflow = open_workflow(CONFIG)
PROJECT, settings = workflow.project, workflow.config
MODE = settings["mode"]
RUN_TRAINING = MODE == "demo" or RUN_RESEARCH_TRAINING
setup_plotting()
display_table(
    ["Control", "Value"],
    [
        ["Software", __version__],
        ["Mode", MODE],
        ["Devices", ", ".join(settings["sources"])],
        [
            "Translation width / epochs",
            f"{settings['translation_config']['model']['width']} / {settings['translation_config']['training']['epochs']}",
        ],
        ["Classifier epochs", settings["classifier_config"]["training"]["epochs"]],
        [
            "Classifier / translation device",
            f"{settings['classifier_device']} / {settings['translation_device']}",
        ],
        ["Research training requested", RUN_RESEARCH_TRAINING],
    ],
)

### Read the execution scope
The demo verifies software behavior. Its plots and scores are synthetic and have no scientific performance interpretation. Research mode uses authorized preprocessed arrays and the recovered width-128 graph. Published-result reproduction remains unresolved; review the paper alignment document before reporting results.

## Dataset access
The [Multi-Instrument SERS Dataset](https://zenodo.org/records/23372983) is hosted on Zenodo with restricted file access. Request access with a brief intended-use statement. After approval, download/extract the files and follow the [local data setup guide](../docs/DATA.md). The synthetic demo requires no dataset access. In a terminal, `sers-pipeline dataset-info` shows the access and configuration steps.


## Steps
### 1. Select a completed model and input path
Run the control center first. Gradients below are with respect to classifier inputs scaled to `[-1,1]`, not the raw instrument signal. Select `logit` or `probability` and a fixed target class; SmoothGrad keeps that class fixed across perturbations.

In [ ]:
import torch
from sersd2d_superraman.classification.model import load_checkpoint
from sersd2d_superraman.explainability import explain

DEVICE = settings["sources"][0]
USE_TRANSLATED = True
TARGET_CLASS = 0
TARGET = "logit"
SMOOTH_SAMPLES = 4
NOISE_STD = 0.01
checkpoint = workflow.output / "classification/checkpoint"
if not (checkpoint / "metadata.json").is_file():
    raise ValueError("Run 00_control_center or 03_train_classifier before explaining a model.")
input_file = (
    workflow.output / "evaluation" / DEVICE / "translated.npz"
    if USE_TRANSLATED
    else workflow.prepared / DEVICE / "test.npz"
)
batch = load_batch(input_file, normalized=not USE_TRANSLATED)
row = int(np.flatnonzero(batch.labels == TARGET_CLASS)[0])
model, metadata = load_checkpoint(checkpoint, settings["classifier_device"])
gradient, classes = explain(
    model,
    batch.spectra[row : row + 1],
    target_classes=np.array([TARGET_CLASS]),
    target=TARGET,
    smooth_samples=SMOOTH_SAMPLES,
    noise_std=NOISE_STD,
    seed=37,
)
assert gradient.shape == (1, 1400) and np.isfinite(gradient).all()

### 2. Display the spectrum and explanation
A gradient indicates sensitivity of this selected score to the input; it does not establish a biochemical mechanism. Demo explanations are synthetic. SHAP and historical BCE-based saliency are outside this release.

In [ ]:
import matplotlib.pyplot as plt
from sersd2d_superraman.data.registry import CLASSES

x = np.arange(1400) if batch.axis_cm1 is None else batch.axis_cm1
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, layout="constrained")
axes[0].plot(x, batch.spectra[row], color="#176B87")
axes[0].set(
    title=f"{MODE.title()} / {DEVICE} / {CLASSES[TARGET_CLASS]} / {TARGET} target", ylabel="Input intensity"
)
axes[1].plot(x, gradient[0], color="#9B3B54")
axes[1].set(
    ylabel="Mean input gradient",
    xlabel="Spectral bin index" if batch.axis_cm1 is None else "Raman shift (cm⁻¹)",
)
plt.show()

### 3. Record explanation provenance

In [ ]:
from sersd2d_superraman.provenance import write_json

explanation_directory = workflow.output / "explanations"
explanation_directory.mkdir(exist_ok=True)
fig.savefig(explanation_directory / "saliency.png", dpi=160)
write_json(
    explanation_directory / "saliency.json",
    {
        "mode": MODE,
        "device": DEVICE,
        "translated": USE_TRANSLATED,
        "sample_id": str(batch.sample_ids[row]),
        "target_class": TARGET_CLASS,
        "target": TARGET,
        "smooth_samples": SMOOTH_SAMPLES,
        "noise_std": NOISE_STD,
        "seed": 37,
        "checkpoint_sha256": metadata["weights_sha256"],
        "gradient_coordinate": "classifier_scaled_input",
        "paper_reproduction_certified": False,
    },
)
{"finite_gradient": bool(np.isfinite(gradient).all()), "target_class": int(classes[0])}

## Checks
Artifact reuse checks package-source and file hashes plus input/configuration identities. Full worker logs are stored under the run directory in logs/. Test rows are evaluated after checkpoint selection; they do not choose the models.

In [ ]:
display_table(["Stage", "Status"], workflow.status().items())

## Next Steps
For real experiments, copy `configs/workflow_research.json` to `configs/local/my_run.json`, set the data path and a new run directory, and select it in `CONFIG`. Set RUN_RESEARCH_TRAINING = True explicitly after reviewing the displayed settings. See the [notebook guide](../docs/NOTEBOOK_GUIDE.md) for stage navigation and interpretation.